In [1]:
# Import Dependencies

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_recall_fscore_support

2025-11-01 09:45:27.470010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761990327.688415      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761990327.744716      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Paths

root = "/kaggle/input/breast-cancer-detection-mri/Breast_Cancer_MRI_Dataset"
IMG_SIZE = (512, 512)
BATCH = 16
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [3]:
# Load dataset

def build_ds(subdir, augment=False, shuffle=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(root, subdir),
        labels="inferred",
        label_mode="int",
        color_mode="rgb",
        batch_size=BATCH,
        image_size=IMG_SIZE,
        shuffle=shuffle,
        seed=SEED
    )

    aug = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.05),
        layers.RandomContrast(0.1),
        layers.RandomZoom(0.1)
    ]) if augment else None

    def _map(x, y):
        x = tf.cast(x, tf.float32)
        if aug is not None:
            x = aug(x, training=True)
        return x, y

    return (ds.map(_map, num_parallel_calls=AUTOTUNE)
              .cache()
              .prefetch(AUTOTUNE))

train_ds = build_ds("train", augment=True, shuffle=True)
val_ds   = build_ds("validation", augment=False, shuffle=False)
test_ds  = build_ds("test", augment=False, shuffle=False)

Found 4000 files belonging to 2 classes.


I0000 00:00:1761990382.629976      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761990382.630671      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 400 files belonging to 2 classes.
Found 400 files belonging to 2 classes.


In [4]:
# Build model

base = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    include_preprocessing = True
)

base.trainable = False  # Stage 1: freeze backbone

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid", kernel_regularizer=regularizers.l2(0.001))(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

/usr/local/lib/python3.11/dist-packages/keras/src/applications/mobilenet_v3.py:452: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# Train head

callbacks = [
    keras.callbacks.ModelCheckpoint("/kaggle/working/MOBILENETV3SMALL.keras", monitor="val_auc", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_auc", patience=10, mode="max", restore_best_weights=True)
]

In [6]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=callbacks, verbose=1)

Epoch 1/100


I0000 00:00:1761990467.559009     112 service.cc:148] XLA service 0x7f2f64034fc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761990467.559784     112 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761990467.559799     112 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761990469.249059     112 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1761990471.028810     112 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1761990471.202121     112 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  3/250 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.3368 - auc: 0.3866 - loss: 0.9003   

I0000 00:00:1761990475.089801     112 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.4943 - auc: 0.4932 - loss: 0.7657
Epoch 1: val_auc improved from -inf to 0.69017, saving model to /kaggle/working/MOBILENETV3SMALL.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 134s 471ms/step - accuracy: 0.4943 - auc: 0.4932 - loss: 0.7656 - val_accuracy: 0.6600 - val_auc: 0.6902 - val_loss: 0.6645
Epoch 2/100
249/250 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5474 - auc: 0.5661 - loss: 0.7085
Epoch 2: val_auc improved from 0.69017 to 0.78237, saving model to /kaggle/working/MOBILENETV3SMALL.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.5475 - auc: 0.5661 - loss: 0.7085 - val_accuracy: 0.7275 - val_auc: 0.7824 - val_loss: 0.6391
Epoch 3/100
249/250 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5838 - auc: 0.6094 - loss: 0.6853
Epoch 3: val_auc improved from 0.78237 to 0.83890, saving model to /kaggle/working/MOBILENETV3SMALL.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.5837 - auc: 0.6094 - loss: 0.68

In [7]:
# Evaluate with best threshold

model.load_weights("/kaggle/working/MOBILENETV3SMALL.keras")

def best_threshold(ds):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    return thr[np.argmax(youden)]

thr = best_threshold(val_ds)
print("Best threshold:", thr)

def metrics_at_threshold(ds, thr):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= thr).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    return dict(auc=auc, precision=prec, recall=rec, f1=f1, specificity=spec, cm=cm)

print("Validation metrics:", metrics_at_threshold(val_ds, thr))
print("Test metrics:", metrics_at_threshold(test_ds, thr))

Best threshold: 0.5308481
Validation metrics: {'auc': 0.9509499999999999, 'precision': 0.8979591836734694, 'recall': 0.88, 'f1': 0.888888888888889, 'specificity': 0.9, 'cm': array([[180,  20],
       [ 24, 176]])}
Test metrics: {'auc': 0.922825, 'precision': 0.8622448979591837, 'recall': 0.845, 'f1': 0.8535353535353536, 'specificity': 0.865, 'cm': array([[173,  27],
       [ 31, 169]])}


## Fine Tuning

In [11]:
model.load_weights("/kaggle/working/MOBILENETV3SMALL.keras")

base.trainable = True # Unfreeze the backbone

callbacks_for_fine_tune = [
    keras.callbacks.ModelCheckpoint("/kaggle/working/MOBILENETV3LARGE.keras", monitor="val_auc", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_auc", patience=15, mode="max", restore_best_weights=True)
]

In [12]:
# recompile with low LR
model.compile(
    optimizer=keras.optimizers.Adam(1e-5), 
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

In [13]:
history2 = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=1)

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9981 - auc: 1.0000 - loss: 0.0275
Epoch 1: val_auc improved from 0.99409 to 0.99410, saving model to /kaggle/working/MOBILENETV3SMALL.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 95ms/step - accuracy: 0.9981 - auc: 1.0000 - loss: 0.0275 - val_accuracy: 0.9750 - val_auc: 0.9941 - val_loss: 0.1075
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.9976 - auc: 1.0000 - loss: 0.0252
Epoch 2: val_auc did not improve from 0.99410
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.9976 - auc: 1.0000 - loss: 0.0252 - val_accuracy: 0.9625 - val_auc: 0.9905 - val_loss: 0.1543
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.9990 - auc: 1.0000 - loss: 0.0225
Epoch 3: val_auc did not improve from 0.99410
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.9990 - auc: 1.0000 - loss: 0.0225 - val_accuracy: 0.9700 - val_auc: 0.9908 - val_loss: 0.1360
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 75m

In [15]:
# Evaluate with best threshold after fine tuning

model.load_weights("/kaggle/working/MOBILENETV3SMALL.keras")

def best_threshold(ds):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    return thr[np.argmax(youden)]

thr = best_threshold(val_ds)
print("Best threshold:", thr)

def metrics_at_threshold(ds, thr):
    y_true = np.concatenate([y for _, y in ds], axis=0)
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= thr).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    return dict(auc=auc, precision=prec, recall=rec, f1=f1, specificity=spec, cm=cm)

print("Validation metrics:", metrics_at_threshold(val_ds, thr))
print("Test metrics:", metrics_at_threshold(test_ds, thr))

Best threshold: 0.40067193
Validation metrics: {'auc': 0.994725, 'precision': 0.9848484848484849, 'recall': 0.975, 'f1': 0.9798994974874371, 'specificity': 0.985, 'cm': array([[197,   3],
       [  5, 195]])}
Test metrics: {'auc': 0.9765, 'precision': 0.9405940594059405, 'recall': 0.95, 'f1': 0.9452736318407959, 'specificity': 0.94, 'cm': array([[188,  12],
       [ 10, 190]])}


In [16]:
!pip install tensorflow_model_optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 5.6 MB/s eta 0:00:00a 0:00:01


In [17]:
import tensorflow_model_optimization as tfmot

In [19]:
# quantize_model = tfmot.quantization.keras.quantize_model

# Load the weights
model.load_weights("/kaggle/working/MOBILENETV3SMALL.keras")

# Define a generator function for the representative dataset
def representative_data_gen():
    for input_value, _ in test_ds.take(100): 
        yield [input_value] # The converter expects a list of arrays

In [20]:
# Create the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set the input and output tensors to uint8 
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

In [21]:
# Convert the model
tflite_quant_model = converter.convert()

Saved artifact at '/tmp/tmp7bzly1cl'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32, name='keras_tensor_180')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139844608994256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608994448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608993488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608995024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608995216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608996176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608997520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608997904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608997712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139844608996368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1398446089

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1761995243.536817      37 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1761995243.536854      37 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1761995243.655168      37 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


In [22]:
# Save the quantized model
quantized_model_path = "/kaggle/working/MOBILENETV3SMALL_8-BIT_PTQ.tflite"
with open(quantized_model_path, "wb") as f:
    f.write(tflite_quant_model)

In [23]:
files = [
    "/kaggle/working/MOBILENETV3SMALL.keras",
    "/kaggle/working/MOBILENETV3SMALL_8-BIT_PTQ.tflite"
]

for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{f}: {size_mb:.2f} MB")

/kaggle/working/MOBILENETV3SMALL.keras: 11.30 MB
/kaggle/working/MOBILENETV3SMALL_8-BIT_PTQ.tflite: 1.16 MB


In [24]:
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
input_type = interpreter.get_input_details()[0]['dtype']
print('input: ', input_type)
output_type = interpreter.get_output_details()[0]['dtype']
print('output: ', output_type)

input:  <class 'numpy.uint8'>
output:  <class 'numpy.uint8'>
